### Block 0 — Install dependencies

In [1]:
!pip install -q datasets huggingface_hub scikit-learn tqdm pandas requests


### Block 1 — Load FiQA-SA dataset (ChanceFocus/fiqa_sa)

In [8]:
import os
import getpass

from datasets import load_dataset
from huggingface_hub import HfFolder

# Use the FinAI FiQA-SA dataset
DATASET_NAME = "TheFinAI/flare-fiqasa"

# Get HF token
hf_token = HfFolder.get_token() or os.getenv("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (hf_...): ")

HfFolder.save_token(hf_token)

# Load all splits for this dataset
ds_all = load_dataset(DATASET_NAME, token=hf_token)

print("Available splits and sizes:")
for split_name, split_ds in ds_all.items():
    print(f"  {split_name}: {len(split_ds)} examples")

# Explicitly choose the test split (should be ~235 examples)
SPLIT = "test"
if SPLIT not in ds_all:
    raise ValueError(f"'test' split not found. Available splits: {list(ds_all.keys())}")

dataset = ds_all[SPLIT]

print(f"\nUsing dataset: {DATASET_NAME}  split: {SPLIT}")
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


Available splits and sizes:
  train: 750 examples
  test: 235 examples
  valid: 188 examples

Using dataset: TheFinAI/flare-fiqasa  split: test
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


### Block 2 — Label config + prediction normalization

In [9]:
import re
from typing import Optional

from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import pandas as pd

LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: Optional[str]) -> str:
    """
    Map a free-form model output to one of the three labels.
    """
    if raw is None:
        return "neutral"
    text = raw.strip().lower()

    # Exact match
    for label in LABELS:
        if text == label:
            return label

    # Whole-word match
    for label in LABELS:
        if re.search(r"\b" + re.escape(label) + r"\b", text):
            return label

    # Fallback
    return "neutral"


### Block 3 — Qwen2.5-32B chat client + single-sentence classifier

In [10]:
import time
from huggingface_hub import InferenceClient

MODEL_ID_QWEN = "Qwen/Qwen2.5-32B-Instruct"

# Reuse the same HF token used for the dataset
client_qwen = InferenceClient(model=MODEL_ID_QWEN, token=hf_token)

SYSTEM_INSTRUCTION = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article or microblog,\n"
    "classify its overall sentiment from the perspective of an investor as\n"
    "exactly one of: negative, neutral, or positive.\n"
    "Respond with only one word: negative, neutral, or positive."
)

def classify_with_qwen(
    sentence: str,
    max_retries: int = 6,
    sleep_base: float = 1.0,
) -> str:
    """
    Call Qwen2.5-32B-Instruct via Hugging Face Inference API
    using the conversational (chat) interface, and return a
    normalized three-way sentiment label.
    """
    last_err = None

    for attempt in range(max_retries):
        try:
            messages = [
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": sentence},
            ]

            resp = client_qwen.chat_completion(
                messages=messages,
                max_tokens=16,
                temperature=0.0,
                stream=False,
            )

            # resp.choices[0].message can be a dict-like object or have .content
            msg = resp.choices[0].message
            content = msg["content"] if isinstance(msg, dict) else msg.content

            # content can be a string or a list of parts
            if isinstance(content, list):
                text_out = "".join(
                    part.get("text", "") if isinstance(part, dict) else str(part)
                    for part in content
                )
            else:
                text_out = str(content)

            return normalize_prediction(text_out)

        except Exception as e:
            last_err = e
            wait = sleep_base * (2 ** attempt)
            print(f"[Qwen2.5-32B] Error on attempt {attempt+1}/{max_retries}: {last_err}")
            time.sleep(wait)

    print("[Qwen2.5-32B] Failed after retries, returning 'neutral'.")
    return "neutral"


### Block 4 — Run evaluation on FiQA-SA

In [11]:
# Optional: limit the number of samples for a quick test
MAX_SAMPLES = None  # set to an int like 50 for a cheaper run

y_true = []
y_pred = []

if MAX_SAMPLES is None:
    eval_dataset = dataset
else:
    eval_dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

print(f"Evaluating model: {MODEL_ID_QWEN} on {len(eval_dataset)} FiQA-SA examples...\n")

for example in tqdm(eval_dataset, total=len(eval_dataset)):
    sentence = example["text"]                       # change if your text column has another name
    true_label = example["answer"].lower().strip()   # change if your label column has another name
    pred_label = classify_with_qwen(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(eval_dataset))
print("Got predictions for:", len(y_pred))


Evaluating model: Qwen/Qwen2.5-32B-Instruct on 235 FiQA-SA examples...



100%|██████████| 235/235 [05:46<00:00,  1.47s/it]

Total examples: 235
Got predictions for: 235


### Block 5 — Metrics + sample predictions and errors

In [12]:
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

df = pd.DataFrame({
    "text": [ex["text"] for ex in eval_dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



Overall accuracy: 0.7149

Classification report:
              precision    recall  f1-score   support

    negative       0.89      0.84      0.86        76
     neutral       0.18      0.67      0.28        18
    positive       0.97      0.65      0.78       141

    accuracy                           0.71       235
   macro avg       0.68      0.72      0.64       235
weighted avg       0.88      0.71      0.77       235


First 10 predictions:
                                                                                                                            text     gold     pred
                                                     $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash. negative negative
                                                                         Legal & General share price: Finance chief to step down negative negative
                                                                 Kingfisher share price slides on cost to